In [1]:
!python3 --version

Python 3.12.12


In [1]:
!pip3 install transformers sentence_transformers faiss-cpu fastapi uvicorn pyngrok tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 36.6 MB/s eta 0:00:00


In [ ]:
# Ejecutar si se trabaja en maquina local
MAIN_ROUTE = '...'

In [2]:
# Ejecutar si se trabaja en google colab
from google.colab import drive
drive.mount('/content/drive')

MAIN_ROUTE = '/content/drive/MyDrive/Proyecto-de-Investigacion'

Mounted at /content/drive


In [13]:
# Cargar tokenizer y modelo de lenguaje localmente
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
#torch.set_num_threads(8)

# Usar modelo desde microsoft
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-4-mini-instruct")
modelo = AutoModelForCausalLM.from_pretrained("microsoft/Phi-4-mini-instruct", dtype=torch.bfloat16).to("cuda")

# Usar modelo desde local
#tokenizer = AutoTokenizer.from_pretrained(MAIN_ROUTE+"/Phi-4-mini-instruct")
#modelo = AutoModelForCausalLM.from_pretrained(MAIN_ROUTE+"/Phi-4-mini-instruct", dtype=torch.bfloat16).to("cuda")


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/249 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.77G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

In [4]:
torch.get_num_threads()

1

In [3]:
from sentence_transformers import SentenceTransformer

# Cargar modelo localmente
embedder = SentenceTransformer(MAIN_ROUTE+"/modelos/all-MiniLM-L6-v2")

In [4]:
import faiss
import numpy as np

embeddings = np.load(MAIN_ROUTE+"/embeddings/CHUNK_PRODUCTO.npy")
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_FICHA.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_DESCRIPCION.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_CARACTERISTICAS.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_OBSERVACIONES.npy")))
#embeddings = np.concatenate((embeddings,np.load("embeddings/CHUNK_RECOMENDACIONES.npy")))

# Crear índice FAISS
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [ ]:
import pandas as pd

df = pd.read_csv(MAIN_ROUTE+"/data/processed/productos_corpus.csv",delimiter=",")
df['CHUNK_DESCRIPCION'] = df['CHUNK_DESCRIPCION'].fillna('')
df['CHUNK_CARACTERISTICAS'] = df['CHUNK_CARACTERISTICAS'].fillna('')
df['CHUNK_OBSERVACIONES'] = df['CHUNK_OBSERVACIONES'].fillna('')
df['CHUNK_RECOMENDACIONES'] = df['CHUNK_RECOMENDACIONES'].fillna('')
tamanio = len(df)

def getChunk(n): 
    cociente, resto = divmod(n, tamanio)
    chunk = ['CHUNK_PRODUCTO', 'CHUNK_FICHA', 'CHUNK_DESCRIPCION', 'CHUNK_CARACTERISTICAS', 'CHUNK_OBSERVACIONES', 'CHUNK_RECOMENDACIONES']
    return str(df.iloc[resto][chunk[cociente]])

In [11]:
getChunk(0)

'ovalin circular para sobreponer vidriotransparente de la marca orange del area de baños para los sanitarios de la linea ovalines de procedencia importado'

In [14]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

def promartBoot(texto,mensajes,k):
    # --------
    pregunta_emb = embedder.encode([texto], convert_to_numpy=True)
    # Buscar los k textos más cercanos
    distancias, indices = index.search(pregunta_emb, k)
    contexto = "\n".join([getChunk(i) for i in indices[0]])
    mensajes.append({"role": "system","content":"CONTEXTO: "+contexto})
    #print(contexto)
    #print('----------------')
    # --------

    mensajes.append({"role": "user","content":texto})

    pipe = pipeline("text-generation", model=modelo, tokenizer=tokenizer)

    generation_args = {
        "max_new_tokens": 500,
        "return_full_text": False,
        "temperature": 0.0,
        "do_sample": False,
    }

    output = pipe(mensajes, **generation_args)
    return output[0]['generated_text'], contexto

In [15]:
resp, contexto = promartBoot('necesito un ovalin circular mimbell',[],3)
print(resp)
print('---------------')
print(contexto)

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Parece que estás buscando un Ovalin circular mimbell, que es un tipo de tapicería o alfombra. Sin embargo, los productos que mencionaste anteriormente (Buzios blanco de la marca Italgrif, padua bl de la marca Vainsa, y Centurion Jr blanco de la marca Urrea) son tapicerías, no alfombras. Si estás buscando una alfombra ovalin circular, necesitarás especificar los detalles como el tamaño, el material y el color que deseas.

Si estás interesado en comprar una alfombra ovalin circular, te recomendaría visitar tiendas de tapicería, tiendas de muebles o tiendas en línea especializadas en alfombras. Proporciona los detalles específicos que necesitas para encontrar el producto adecuado.
---------------
ovalin circular para sobreponer buzios blanco de la marca italgrif del area de baños para los sanitarios de la linea ovalines de procedencia nacional
ovalin circular para sobreponer padua bl de la marca vainsa del area de baños para los sanitarios de la linea ovalines de procedencia nacional
oval

In [ ]:
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

# Paso 1: Autenticarse con Hugging Face
# Reemplaza TU_TOKEN aquí con tu token de Hugging Face
login(token="")

# Paso 2: Cargar modelo y tokenizer (usa CPU con float32)
model_id = "google/gemma-2b-it"

print("⏳ Cargando modelo Gemma...")

tokenizer = AutoTokenizer.from_pretrained(model_id, token=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto",
    token=True
)

# Paso 3: Crear pipeline de generación de texto
#chat = pipeline("text-generation", model=model, tokenizer=tokenizer, device=-1)
chat = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Paso 4: Iniciar bucle de conversación
historial = []

def construir_prompt(conversacion):
    prompt = ""
    for entrada in conversacion:
        rol = entrada["role"]
        contenido = entrada["content"]
        prompt += f"<start_of_turn>{rol}\n{contenido}<end_of_turn>\n"
    prompt += "<start_of_turn>model\n"  # Inicia respuesta del modelo
    return prompt

⏳ Cargando modelo Gemma...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/34.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Device set to use cuda:0


In [7]:
user_input = 'sabes que es un martillo?'

historial.append({"role": "user", "content": user_input})
prompt = construir_prompt(historial)

respuesta = chat(
    prompt,
    max_new_tokens=200,
    do_sample=True,
    temperature=0.7,
    pad_token_id=tokenizer.eos_token_id,
)[0]["generated_text"]

# Extraer solo la respuesta del modelo (entre model y end_of_turn)
if "<start_of_turn>model" in respuesta:
    respuesta = respuesta.split("<start_of_turn>model")[-1]
respuesta = respuesta.strip().split("<end_of_turn>")[0].strip()

print(f"Gemma: {respuesta}\n")
historial.append({"role": "model", "content": respuesta})

Gemma: Un martillo es un instrumento de herramientas con una cabeza de martillo. Se utiliza para martillar materiales como madera, piedra o metal.



In [8]:
def promartBootGema(texto,mensajes,k):
    # --------
    pregunta_emb = embedder.encode([texto], convert_to_numpy=True)
    # Buscar los k textos más cercanos
    distancias, indices = index.search(pregunta_emb, k)
    contexto = "\n".join([getChunk(i) for i in indices[0]])
    mensajes.append({"role": "system","content":"CONTEXTO: "+contexto})


    mensajes.append({"role": "user","content":texto})

    prompt = construir_prompt(mensajes)

    respuesta = chat(
        prompt,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
    )[0]["generated_text"]

    # Extraer solo la respuesta del modelo (entre model y end_of_turn)
    if "<start_of_turn>model" in respuesta:
        respuesta = respuesta.split("<start_of_turn>model")[-1]
    respuesta = respuesta.strip().split("<end_of_turn>")[0].strip()

    return str(respuesta)

In [9]:
rep = promartBootGema('necesito un ovalin circular mimbell',[],3)
print(rep)

**Ovalin circular**

**Descripción:**

* Color blanco
* Diámetro: 10 cm
* Material: papel

**Uso:**

* Overponer sobre cualquier superficie horizontal o curva, como la superficie de los baños para los sanitarios de la linea ovalines de procedencia nacional.
* Enciar el ovalin sobre la superficie y presionar para asegurar una buena aderencia.

**Consejos:**

* Reforzar el ovalin con una cinta o papel forte antes de presionarlo.
* Ajustar el tamaño del ovalin para adaptarse a las necesidades específicas del espacio.
* Limitar el número de ovalin sobreponer para evitar un efecto visual desproporcionado.


In [10]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import nest_asyncio
import uvicorn


In [11]:
nest_asyncio.apply()
app = FastAPI()

app.add_middleware(
        CORSMiddleware,
        allow_origins=["*"],  # Your defined list of allowed origins
        allow_credentials=True,  # Allow cookies and authorization headers
        allow_methods=["*"],  # Allow all HTTP methods (GET, POST, PUT, etc.)
        allow_headers=["*"],  # Allow all headers
    )

class Body(BaseModel):
    pregunta: str
    k: int
    history: list

@app.post("/clasificar")
def clasificar(body: Body):
    texto = body.pregunta
    k = body.k
    history = body.history
    respuesta, contexto = promartBoot(texto,history,k)
    #return {"respuesta": respuesta, "contexto": contexto}
    return {"respuesta": respuesta}

@app.post("/clasificar-gema")
def clasificar(body: Body):
    texto = body.pregunta
    k = body.k
    history = body.history
    respuesta = promartBootGema(texto,history,k)
    #return {"respuesta": respuesta, "contexto": contexto}
    return {"respuesta": respuesta}


#uvicorn.run(app, port=8000)

In [12]:
from pyngrok import ngrok

# Inicia el tunel ngrok
ngrok.set_auth_token("1o4ItumAvMRlaYTG9dxyHKNInZq_48f9Txagc76SkaUUpoZB4")
public_url = ngrok.connect(8000)
print(f"🔗 API disponible en: {public_url}")

🔗 API disponible en: NgrokTunnel: "https://af37b181d6db.ngrok-free.app" -> "http://localhost:8000"


In [13]:
import threading
import uvicorn

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run, daemon=True).start()

In [14]:
from datetime import datetime, timezone, timedelta
gmt_minus_5 = timezone(timedelta(hours=-5))
fecha_hora_actual = datetime.now(gmt_minus_5)

print("UPDATE:", fecha_hora_actual.strftime("%Y-%m-%d %H:%M:%S"))

UPDATE: 2025-10-23 19:50:52
